Given customer, contract, and vehicle information available at pricing time, predict motor insurance premium.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import ParameterSampler
from scipy.stats import randint, uniform
from sklearn.inspection import permutation_importance
import shap

In [ ]:
from pathlib import Path

data_files = sorted(Path("../data/raw").glob("motor_vehicle_insurance_part_*.csv.gz"))
if not data_files:
    raise FileNotFoundError("Public dataset parts were not found in ../data/raw")
data = pd.concat((pd.read_csv(path) for path in data_files), ignore_index=True)

In [ ]:
df = data.copy()

print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# -----------------------------
# variable groups
# -----------------------------
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

target_var = "Premium"

date_vars = [
    "Date_birth",
    "Date_driving_licence"
]

categorical_vars = [
    "Distribution_channel",
    "Payment",
    "Type_risk",
    "Area",
    "Second_driver",
    "Type_fuel"
]

discrete_vars = [
    "Seniority",
    "Policies_in_force",
    "Max_policies",
    "Max_products",
    "N_doors"
]

continuous_vars = [
    "Year_matriculation",
    "Power",
    "Cylinder_capacity",
    "Value_vehicle",
    "Length",
    "Weight"
]

id_time_vars = [
    "ID",
    "Date_start_contract",
    "Date_last_renewal",
    "Date_next_renewal"
]

all_focus_vars = id_time_vars + date_vars + categorical_vars + discrete_vars + continuous_vars + [target_var]
all_focus_vars = [c for c in all_focus_vars if c in df.columns]

In [ ]:
date_like_cols = [
    "Date_start_contract",
    "Date_last_renewal",
    "Date_next_renewal",
    "Date_lapse",
    "Date_birth",
    "Date_driving_licence"
]

for col in date_like_cols:
    df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

In [ ]:
def quick_col_check(data, col):
    s = data[col]
    print("=" * 80)
    print(f"COLUMN: {col}")
    print("=" * 80)
    print("dtype:", s.dtype)
    print("missing:", s.isna().sum())
    print("missing %:", round(s.isna().mean() * 100, 2))
    print("nunique:", s.nunique(dropna=True))
    print()

def inspect_categorical(data, col, top_n=50):
    quick_col_check(data, col)
    
    print("Value counts:")
    display(data[col].value_counts(dropna=False).head(top_n))
    
    print("\nSorted unique values:")
    uniq = pd.Series(data[col].dropna().astype(str).unique()).sort_values()
    display(uniq.to_frame(name=col).reset_index(drop=True))

def inspect_numeric(data, col, low_n=20, high_n=20):
    quick_col_check(data, col)
    
    print("Summary stats:")
    display(data[col].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame(name=col))
    
    print(f"\nSmallest {low_n} values:")
    display(data[[col]].sort_values(col, ascending=True).head(low_n))
    
    print(f"\nLargest {high_n} values:")
    display(data[[col]].sort_values(col, ascending=False).head(high_n))

def show_rows(data, condition, cols=None, n=30, sort_by=None, ascending=True):
    subset = data.loc[condition].copy()
    
    if sort_by is not None and sort_by in subset.columns:
        subset = subset.sort_values(sort_by, ascending=ascending)
    
    if cols is not None:
        cols = [c for c in cols if c in subset.columns]
        subset = subset[cols]
    
    print("Rows found:", len(subset))
    display(subset.head(n))

def inspect_extreme_rows(data, col, context_cols=None, n=20):
    if context_cols is None:
        context_cols = []
    
    cols = ["ID", col] + context_cols
    cols = [c for c in cols if c in data.columns]
    
    print(f"\nLowest {n} rows for {col}")
    display(data[cols].sort_values(col, ascending=True).head(n))
    
    print(f"\nHighest {n} rows for {col}")
    display(data[cols].sort_values(col, ascending=False).head(n))

In [ ]:
for col in categorical_vars:
    if col in df.columns:
        inspect_categorical(df, col)

In [ ]:
for col in discrete_vars:
    if col in df.columns:
        inspect_numeric(df, col)
        
        print(f"Value counts for {col}:")
        display(df[col].value_counts(dropna=False).sort_index())

In [ ]:
for col in continuous_vars:
    if col in df.columns:
        inspect_numeric(df, col)

        print(f"Value counts for {col}:")
        display(df[col].value_counts(dropna=False).sort_index())

In [ ]:
context_vehicle = [
    "ID", "Date_last_renewal", "Year_matriculation", "Power", "Cylinder_capacity",
    "Value_vehicle", "N_doors", "Type_fuel", "Length", "Weight", "Premium"
]
context_vehicle = [c for c in context_vehicle if c in df.columns]

show_rows(df, df["N_doors"] <= 0, cols=context_vehicle, n=50)
show_rows(df, df["Power"] <= 0, cols=context_vehicle, n=50)
show_rows(df, df["Cylinder_capacity"] <= 0, cols=context_vehicle, n=50)
show_rows(df, df["Value_vehicle"] <= 0, cols=context_vehicle, n=50)
show_rows(df, df["Length"] <= 0, cols=context_vehicle, n=50)
show_rows(df, df["Weight"] <= 0, cols=context_vehicle, n=50)

In [ ]:
show_rows(df, df["Year_matriculation"].isna(), cols=context_vehicle, n=30)

if "Date_last_renewal" in df.columns:
    show_rows(
        df,
        df["Year_matriculation"] > df["Date_last_renewal"].dt.year,
        cols=context_vehicle,
        n=50
    )

show_rows(df, df["Year_matriculation"] < 1950, cols=context_vehicle, n=50)

In [ ]:
context_premium = [
    "ID", "Date_last_renewal", "Distribution_channel", "Payment",
    "Type_risk", "Area", "Power", "Value_vehicle"
]
context_premium = [c for c in context_premium if c in df.columns]

show_rows(df, df["Premium"] <= 0, cols=["ID", "Date_last_renewal", "Premium"], n=50)
inspect_extreme_rows(df, "Premium", context_cols=context_premium, n=30)

In [ ]:
for col in date_vars + ["Date_last_renewal", "Date_start_contract", "Date_next_renewal"]:
    if col in df.columns:
        quick_col_check(df, col)
        print("Min:", df[col].min())
        print("Max:", df[col].max())
        print()

In [ ]:
df["age_at_renewal"] = (df["Date_last_renewal"] - df["Date_birth"]).dt.days / 365.25
df["licence_tenure"] = (df["Date_last_renewal"] - df["Date_driving_licence"]).dt.days / 365.25
df["age_when_licensed"] = (df["Date_driving_licence"] - df["Date_birth"]).dt.days / 365.25
df["vehicle_age"] = df["Date_last_renewal"].dt.year - df["Year_matriculation"]

date_context = [
    "ID", "Date_birth", "Date_driving_licence", "Date_last_renewal",
    "age_at_renewal", "licence_tenure", "age_when_licensed", "vehicle_age", "Premium"
]
date_context = [c for c in date_context if c in df.columns]

show_rows(df, df["Date_driving_licence"] < df["Date_birth"], cols=date_context, n=50)
show_rows(df, df["age_at_renewal"] < 16, cols=date_context, n=50)
show_rows(df, df["age_at_renewal"] > 100, cols=date_context, n=50)
show_rows(df, df["licence_tenure"] < 0, cols=date_context, n=50)
show_rows(df, df["age_when_licensed"] < 16, cols=date_context, n=50)
show_rows(df, df["vehicle_age"] < 0, cols=date_context, n=50)
show_rows(df, df["vehicle_age"] > 50, cols=date_context, n=50)

In [ ]:
relationship_context = [
    "ID", "Date_last_renewal", "Seniority", "Policies_in_force",
    "Max_policies", "Max_products", "Payment", "Premium"
]
relationship_context = [c for c in relationship_context if c in df.columns]

show_rows(df, df["Seniority"] < 0, cols=relationship_context, n=50)
show_rows(df, df["Policies_in_force"] < 0, cols=relationship_context, n=50)
show_rows(df, df["Max_policies"] < 0, cols=relationship_context, n=50)
show_rows(df, df["Max_products"] < 0, cols=relationship_context, n=50)
show_rows(df, df["Max_policies"] < df["Policies_in_force"], cols=relationship_context, n=50)

In [ ]:
missing_tbl = df[all_focus_vars].isna().sum().sort_values(ascending=False)
display(missing_tbl.to_frame("missing_n"))

In [ ]:
pseudo_missing_tokens = {
    "", " ", "  ", "na", "n/a", "nan", "null", "none", "unknown", "unk", "?", "-", "--"
}

def find_pseudo_missing(data, cols):
    results = []
    
    for col in cols:
        s = data[col]
        
        # convert to stripped lowercase strings, but keep actual NaN separate
        s_str = s.astype(str).str.strip().str.lower()
        count = s_str.isin(pseudo_missing_tokens).sum()
        
        results.append({
            "variable": col,
            "pseudo_missing_n": count
        })
    
    return pd.DataFrame(results).sort_values("pseudo_missing_n", ascending=False)

obj_like_cols = [c for c in all_focus_vars if df[c].dtype == "object"]
display(find_pseudo_missing(df, obj_like_cols))

In [ ]:
for col in obj_like_cols:
    print(f"\n===== {col} =====")
    vals = pd.Series(df[col].dropna().astype(str).str.strip().unique()).sort_values()
    display(vals.to_frame(name=col).reset_index(drop=True).head(100))

In [ ]:
special_codes = [0, -1, 99, 999, 9999]

for col in discrete_vars + continuous_vars:
    if col in df.columns:
        print(f"\n===== {col} =====")
        print(df[col].value_counts(dropna=False).sort_index().head(20))
        for code in special_codes:
            cnt = (df[col] == code).sum()
            if cnt > 0:
                print(f"value == {code}: {cnt}")

Step 2

In [ ]:
text_like_cols = ["Distribution_channel", "Type_fuel"]

for col in text_like_cols:
        df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())

pseudo_missing_tokens = {
    "", " ", "  ", "NA", "N/A", "NaN", "nan", "NULL", "null", "None", "none",
    "UNKNOWN", "unknown", "UNK", "unk", "?", "-", "--"
}

for col in text_like_cols:
        df[col] = df[col].replace(list(pseudo_missing_tokens), np.nan)

In [ ]:
df["Distribution_channel"] = df["Distribution_channel"].replace({
    "0": "0",
    "0.0": "0",
    "00": "0",
    "1": "1",
    "1.0": "1",
    "01": "1",
    "00/01/1900": np.nan
})
    
df.loc[df["Distribution_channel"].astype(str).str.strip().eq("1900-01-00"), "Distribution_channel"] = np.nan
df["Distribution_channel"] = df["Distribution_channel"].astype("object")

In [ ]:
df["Type_fuel"] = df["Type_fuel"].str.upper()
invalid_fuel_mask = ~df["Type_fuel"].isin(["D", "P"]) & df["Type_fuel"].notna()
df.loc[invalid_fuel_mask, "Type_fuel"] = np.nan

In [ ]:
df["flag_n_doors_zero"] = (df["N_doors"] == 0).astype(int)
df["flag_power_missing"] = (df["Power"].isna() | (df["Power"] == 0)).astype(int)
df["flag_type_fuel_missing"] = df["Type_fuel"].isna().astype(int)
df["flag_length_missing"] = df["Length"].isna().astype(int)

In [ ]:
df.loc[df["Power"] == 0, "Power"] = np.nan

In [ ]:
bad_max_policies = df["Max_policies"] < df["Policies_in_force"]
df.loc[bad_max_policies, "Max_policies"] = np.nan

In [ ]:
df["age_at_renewal"] = (df["Date_last_renewal"] - df["Date_birth"]).dt.days / 365.25
df["licence_tenure"] = (df["Date_last_renewal"] - df["Date_driving_licence"]).dt.days / 365.25
df["age_when_licensed"] = (df["Date_driving_licence"] - df["Date_birth"]).dt.days / 365.25
df["vehicle_age"] = df["Date_last_renewal"].dt.year - df["Year_matriculation"]
df["policy_tenure"] = (df["Date_last_renewal"] - df["Date_start_contract"]).dt.days / 365.25

In [ ]:
df["flag_negative_licence_tenure"] = (df["licence_tenure"] < 0).astype(int)
neg_lic_mask = df["licence_tenure"] < 0
df.loc[neg_lic_mask, "licence_tenure"] = np.nan
df.loc[neg_lic_mask, "age_when_licensed"] = np.nan

In [ ]:
if "policy_tenure" in df.columns:
    df.loc[df["policy_tenure"] < 0, "policy_tenure"] = np.nan

In [ ]:
step2_check_cols = [
    "Distribution_channel", "Type_fuel", "Power", "Max_policies",
    "age_at_renewal", "licence_tenure", "age_when_licensed",
    "vehicle_age", "policy_tenure",
    "flag_n_doors_zero", "flag_power_missing",
    "flag_type_fuel_missing", "flag_length_missing",
    "flag_negative_licence_tenure"
]
step2_check_cols = [c for c in step2_check_cols if c in df.columns]

for col in step2_check_cols:
    print(f"\n===== {col} =====")
    print("dtype:", df[col].dtype)
    print("missing:", df[col].isna().sum())
    print("missing %:", round(df[col].isna().mean() * 100, 2))
    
    if df[col].dtype == "object":
        display(df[col].value_counts(dropna=False))
    else:
        display(df[col].describe())
        if df[col].nunique(dropna=False) <= 10:
            print(df[col].value_counts(dropna=False).sort_index())

In [ ]:
# focused validation after Step 2
show_rows(
    df,
    df["Distribution_channel"].isna(),
    cols=["ID", "Date_last_renewal", "Distribution_channel"],
    n=30
)

show_rows(
    df,
    df["flag_power_missing"] == 1,
    cols=["ID", "Date_last_renewal", "Power", "Type_fuel", "Length", "N_doors"],
    n=30
)

show_rows(
    df,
    df["flag_negative_licence_tenure"] == 1,
    cols=[
        "ID", "Date_birth", "Date_driving_licence", "Date_last_renewal",
        "licence_tenure", "age_when_licensed"
    ],
    n=30
)

show_rows(
    df,
    df["Max_policies"].isna() & df["Policies_in_force"].notna(),
    cols=["ID", "Date_last_renewal", "Policies_in_force", "Max_policies"],
    n=30
)

Step3: Deterministic Feature Engineering 

In [ ]:
df["renewal_year"] = df["Date_last_renewal"].dt.year
df["renewal_month"] = df["Date_last_renewal"].dt.month
df["renewal_quarter"] = df["Date_last_renewal"].dt.quarter

In [ ]:
inspect_numeric(df, "Value_vehicle")

In [ ]:
'''# domain-style threshold flags
df["flag_young_driver"] = (df["age_at_renewal"] < 25).astype(int)
df["flag_senior_driver"] = (df["age_at_renewal"] >= 70).astype(int)

# deterministic threshold
df["flag_newly_licensed"] = (df["licence_tenure"] < 2).astype(int)

# keep missing-safe behavior:
for col in ["age_at_renewal", "licence_tenure"]:
    if col in df.columns:
        miss_mask = df[col].isna()
        if col == "age_at_renewal":
            df.loc[miss_mask, ["flag_young_driver", "flag_senior_driver"]] = np.nan
        if col == "licence_tenure":
            df.loc[miss_mask, "flag_newly_licensed"] = np.nan'''

In [ ]:
df["flag_young_driver"] = (df["age_at_renewal"] < 25).astype(int)
df["flag_senior_driver"] = (df["age_at_renewal"] >= 70).astype(int)

df["flag_licence_tenure_missing"] = df["licence_tenure"].isna().astype(int)

df["flag_newly_licensed"] = (
    df["licence_tenure"].notna() & (df["licence_tenure"] < 2)
).astype(int)

In [ ]:
def safe_divide(num, den):
    """
    Elementwise division that returns NaN when denominator is 0 or missing.
    """
    den = den.replace(0, np.nan)
    return num / den

In [ ]:
skewness_check_columns = ["Power", "Cylinder_capacity", "Weight", "Length", "Value_vehicle"]
for col in skewness_check_columns:
    print(f"Skewness of {col}: {df[col].skew()}")

In [ ]:
df["vehicle_value_log"] = np.log(df["Value_vehicle"])
df["vehicle_age_plus_1"] = df["vehicle_age"] + 1

# intensity / ratio features
df["power_weight_ratio"] = safe_divide(df["Power"], df["Weight"])
df["engine_cc_per_power"] = safe_divide(df["Cylinder_capacity"], df["Power"])
df["value_per_vehicle_year"] = safe_divide(df["Value_vehicle"], df["vehicle_age_plus_1"])

df["weight_log"] = np.log(df["Weight"])
df["length_log"] = np.log1p(df["Length"])

In [ ]:
# headroom within observed relationship fields
df["policy_headroom"] = df["Max_policies"] - df["Policies_in_force"]

In [ ]:
numeric_vars = [
    "Seniority",
    "Policies_in_force",
    "Max_policies",
    "Max_products",
    "Year_matriculation",
    "Power",
    "Cylinder_capacity",
    "Value_vehicle",
    "N_doors",
    "Length",
    "Weight",
    "age_at_renewal",
    "licence_tenure",
    "age_when_licensed",
    "vehicle_age",
    "policy_tenure",
    "renewal_year",
    "renewal_month",
    "renewal_quarter"
]

engineered_numeric_vars = [
    "vehicle_value_log",
    "power_weight_ratio",
    "engine_cc_per_power",
    "value_per_vehicle_year",
    "weight_log",
    "length_log",
    "policy_headroom"
]

flag_features = [
    "flag_n_doors_zero",
    "flag_power_missing",
    "flag_type_fuel_missing",
    "flag_length_missing",
    "flag_negative_licence_tenure",
    "flag_licence_tenure_missing",
    "flag_young_driver",
    "flag_senior_driver",
    "flag_newly_licensed",
]

In [ ]:
final_feature_cols = (
    categorical_vars
    + numeric_vars
    + engineered_numeric_vars
    + flag_features
)

# keep only columns that truly exist
final_feature_cols = [c for c in final_feature_cols if c in df.columns]

In [ ]:
print("\nNumber of final features:", len(final_feature_cols))
display(pd.DataFrame({
    "feature": final_feature_cols,
    "dtype": [df[c].dtype for c in final_feature_cols]
}))

step4: Data Split

In [ ]:
time_col = "Date_last_renewal"

required_cols = final_feature_cols + [target_var, time_col, "ID"]
model_df = df[required_cols].copy()

# keep only rows with non-missing target and split timestamp
model_df = model_df.loc[
    model_df[target_var].notna() &
    model_df[time_col].notna()
].copy()

# sort in true chronological order
model_df = model_df.sort_values([time_col, "ID"]).reset_index(drop=True)

print("Modeling rows:", len(model_df))
print("Unique IDs:", model_df["ID"].nunique())
print(f"{time_col} range:", model_df[time_col].min(), "to", model_df[time_col].max())

In [ ]:
def chronological_date_block_split(
    data,
    time_col,
    train_size=0.6,
    valid_size=0.2,
    test_size=0.2
):
    if not np.isclose(train_size + valid_size + test_size, 1.0):
        raise ValueError("train_size + valid_size + test_size must sum to 1.")

    data = data.sort_values([time_col, "ID"]).reset_index(drop=True).copy()

    unique_dates = np.array(sorted(data[time_col].dropna().unique()))
    n_dates = len(unique_dates)

    train_end = int(np.floor(n_dates * train_size))
    valid_end = int(np.floor(n_dates * (train_size + valid_size)))

    train_dates = unique_dates[:train_end]
    valid_dates = unique_dates[train_end:valid_end]
    test_dates = unique_dates[valid_end:]

    train_df = data.loc[data[time_col].isin(train_dates)].copy()
    valid_df = data.loc[data[time_col].isin(valid_dates)].copy()
    test_df = data.loc[data[time_col].isin(test_dates)].copy()

    return train_df, valid_df, test_df, train_dates, valid_dates, test_dates


train_df, valid_df, test_df, train_dates, valid_dates, test_dates = chronological_date_block_split(
    data=model_df,
    time_col=time_col,
    train_size=0.6,
    valid_size=0.2,
    test_size=0.2
)

In [ ]:
def summarize_split(split_name, split_df, time_col, target_var):
    print(f"\n===== {split_name.upper()} =====")
    print("rows:", len(split_df))
    print("unique IDs:", split_df["ID"].nunique())
    print("unique dates:", split_df[time_col].nunique())
    print(f"{time_col} range:", split_df[time_col].min(), "to", split_df[time_col].max())
    print(f"{target_var} missing:", split_df[target_var].isna().sum())


summarize_split("train", train_df, time_col, target_var)
summarize_split("valid", valid_df, time_col, target_var)
summarize_split("test", test_df, time_col, target_var)

In [ ]:
X_train = train_df[final_feature_cols].copy()
y_train = train_df[target_var].copy()

X_valid = valid_df[final_feature_cols].copy()
y_valid = valid_df[target_var].copy()

X_test = test_df[final_feature_cols].copy()
y_test = test_df[target_var].copy()

print("\nShapes")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

In [ ]:
train_ids = set(train_df["ID"].unique())
valid_ids = set(valid_df["ID"].unique())
test_ids = set(test_df["ID"].unique())

print("\nID overlap diagnostics")
print("train ∩ valid:", len(train_ids & valid_ids))
print("train ∩ test :", len(train_ids & test_ids))
print("valid ∩ test :", len(valid_ids & test_ids))

Step5: Training Only EDA

In [ ]:
eda_df = train_df.copy()

print("EDA rows:", len(eda_df))
print("EDA unique IDs:", eda_df["ID"].nunique())
print("Target:", target_var)

In [ ]:
categorical_features = [
    col for col in [
        "Distribution_channel",
        "Payment",
        "Type_risk",
        "Area",
        "Second_driver",
        "Type_fuel"
    ]
]

'''flag_features = [
    col for col in [
        "flag_n_doors_zero",
        "flag_power_missing",
        "flag_type_fuel_missing",
        "flag_length_missing",
        "flag_negative_licence_tenure",
        "flag_licence_tenure_missing",
        "flag_young_driver",
        "flag_senior_driver",
        "flag_newly_licensed"
    ]
]'''

numeric_features = [
    col for col in final_feature_cols
    if col not in categorical_features + flag_features
]

print("Number of categorical features:", len(categorical_features))
print("Number of numeric features:", len(numeric_features))
print("Number of flag features:", len(flag_features))

In [ ]:
target_summary = eda_df[target_var].describe().to_frame().T
display(target_summary)

plt.figure(figsize=(8, 4))
plt.hist(eda_df[target_var].dropna(), bins=50)
plt.title(f"Training Target Distribution: {target_var}")
plt.xlabel(target_var)
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8, 4))
plt.boxplot(eda_df[target_var].dropna(), vert=False)
plt.title(f"Training Target Boxplot: {target_var}")
plt.xlabel(target_var)
plt.show()

In [ ]:
missing_summary = (
    eda_df[final_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
)

missing_summary_nonzero = missing_summary.loc[missing_summary["missing_rate"] > 0].copy()
display(missing_summary_nonzero)

plt.figure(figsize=(10, max(4, 0.35 * len(missing_summary_nonzero))))
plt.barh(missing_summary_nonzero["feature"], missing_summary_nonzero["missing_rate"])
plt.title("Training-Set Missingness Rate by Feature")
plt.xlabel("Missing Rate")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
numeric_summary = eda_df[numeric_features].describe().T
numeric_summary["missing_rate"] = eda_df[numeric_features].isna().mean()
numeric_summary["n_unique"] = eda_df[numeric_features].nunique(dropna=True)

display(numeric_summary.sort_values("missing_rate", ascending=False))

In [ ]:
def summarize_categorical_feature(data, col, target_col):
    summary = (
        data.groupby(col, dropna=False)[target_col]
        .agg(["count", "mean", "median"])
        .reset_index()
        .sort_values("count", ascending=False)
    )
    return summary

for col in categorical_features + flag_features:
    print(f"\n===== {col} =====")
    display(summarize_categorical_feature(eda_df, col, target_var))

In [ ]:
key_numeric_for_plots = [
    col for col in [
        "Value_vehicle",
        "vehicle_value_log",
        "Power",
        "Weight",
        "Length",
        "age_at_renewal",
        "licence_tenure",
        "vehicle_age",
        "policy_tenure",
        "Policies_in_force",
        "Max_policies",
        "policy_headroom",
        "power_weight_ratio",
        "engine_cc_per_power",
        "value_per_vehicle_year"
    ]
    if col in eda_df.columns
]

for col in key_numeric_for_plots:
    plt.figure(figsize=(8, 4))
    plt.hist(eda_df[col].dropna(), bins=40)
    plt.title(f"Training Distribution: {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [ ]:
corr_candidates = numeric_features.copy()

pearson_corr = (
    eda_df[corr_candidates + [target_var]]
    .corr(numeric_only=True)[target_var]
    .drop(target_var)
    .sort_values(key=np.abs, ascending=False)
    .rename("pearson_corr_with_target")
    .reset_index()
    .rename(columns={"index": "feature"})
)

display(pearson_corr)

top_numeric_features = pearson_corr["feature"].head(10).tolist()

for col in top_numeric_features:
    plot_df = eda_df[[col, target_var]].dropna()
    
    plt.figure(figsize=(6, 4))
    plt.scatter(plot_df[col], plot_df[target_var], s=10, alpha=0.35)
    plt.title(f"{target_var} vs {col} (train only)")
    plt.xlabel(col)
    plt.ylabel(target_var)
    plt.show()

In [ ]:
for col in categorical_features + flag_features:
    plot_df = eda_df[[col, target_var]].copy()
    
    # convert to string only for plotting readability
    plot_df[col] = plot_df[col].astype("object").where(plot_df[col].notna(), "Missing")
    
    group_order = (
        plot_df.groupby(col)[target_var]
        .median()
        .sort_values()
        .index
    )
    
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=plot_df, x=col, y=target_var, order=group_order)
    plt.title(f"{target_var} by {col} (train only)")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
corr_matrix = eda_df[numeric_features].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Training-Set Correlation Heatmap: Numeric Features")
plt.show()

# high-correlation pairs
corr_pairs = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (
    corr_pairs.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "corr"})
)

high_corr_pairs["abs_corr"] = high_corr_pairs["corr"].abs()
high_corr_pairs = high_corr_pairs.loc[high_corr_pairs["abs_corr"] >= 0.80].sort_values(
    "abs_corr", ascending=False
)

display(high_corr_pairs)

Step6: Preprocessing Design

In [ ]:
FEATURE_GROUPS = {
    "categorical": [
        "Distribution_channel",
        "Type_risk",
        "Type_fuel"
    ],
    "binary": [
        "Payment",
        "Area",
        "Second_driver",
        "flag_n_doors_zero",
        "flag_power_missing",
        "flag_type_fuel_missing",
        "flag_length_missing",
        "flag_negative_licence_tenure",
        "flag_licence_tenure_missing",
        "flag_young_driver",
        "flag_senior_driver",
        "flag_newly_licensed"
    ],
    "numeric_tree": [
        "Seniority",
        "Policies_in_force",
        "Max_policies",
        "Max_products",
        "Year_matriculation",
        "Power",
        "Cylinder_capacity",
        "Value_vehicle",
        "N_doors",
        "Length",
        "Weight",
        "age_at_renewal",
        "licence_tenure",
        "age_when_licensed",
        "vehicle_age",
        "policy_tenure",
        "renewal_year",
        "renewal_month",
        "renewal_quarter",
        "vehicle_value_log",
        "power_weight_ratio",
        "engine_cc_per_power",
        "value_per_vehicle_year",
        "weight_log",
        "length_log",
        "policy_headroom"
    ],
    "numeric_linear": [
        "Seniority",
        "Policies_in_force",
        "Max_policies",
        "Max_products",
        "Power",
        "Cylinder_capacity",
        "N_doors",
        "age_at_renewal",
        "licence_tenure",
        "vehicle_age",
        "policy_tenure",
        "renewal_year",
        "renewal_month",
        "vehicle_value_log",
        "power_weight_ratio",
        "engine_cc_per_power",
        "value_per_vehicle_year",
        "weight_log",
        "length_log",
        "policy_headroom"
    ]
}

In [ ]:
FEATURE_VIEWS = {
    "linear": {
        "numeric": FEATURE_GROUPS["numeric_linear"],
        "categorical": FEATURE_GROUPS["categorical"],
        "binary": FEATURE_GROUPS["binary"],
        "scale_numeric": True,
        "drop_first": True
    },
    "tree": {
        "numeric": FEATURE_GROUPS["numeric_tree"],
        "categorical": FEATURE_GROUPS["categorical"],
        "binary": FEATURE_GROUPS["binary"],
        "scale_numeric": False,
        "drop_first": False
    }
}

def inspect_feature_view(view_name, view_config, reference_cols):
    all_cols = (
        view_config["numeric"]
        + view_config["categorical"]
        + view_config["binary"]
    )

    duplicates = pd.Series(all_cols).value_counts()
    duplicates = duplicates[duplicates > 1]

    missing_from_reference = [c for c in all_cols if c not in reference_cols]

    print(f"\n===== {view_name.upper()} FEATURE VIEW =====")
    print("Total columns listed :", len(all_cols))
    print("Unique columns       :", len(set(all_cols)))
    print("Duplicates found     :", "None" if duplicates.empty else "")
    if not duplicates.empty:
        print(duplicates)

    print("Missing from X_train :", "None" if len(missing_from_reference) == 0 else "")
    if len(missing_from_reference) > 0:
        print(missing_from_reference)


for view_name, view_config in FEATURE_VIEWS.items():
    inspect_feature_view(view_name, view_config, X_train.columns)

In [ ]:
def make_preprocessor(view_config):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()) if view_config["scale_numeric"] else ("identity", "passthrough")
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            drop="first" if view_config["drop_first"] else None
        ))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, view_config["numeric"]),
            ("cat", categorical_pipe, view_config["categorical"]),
            ("bin", "passthrough", view_config["binary"])
        ],
        remainder="drop"
    )

    return preprocessor

In [ ]:
MODEL_SPECS = {
    "ols": {
        "family": "linear",
        "estimator": LinearRegression()
    },
    "ridge": {
        "family": "linear",
        "estimator": Ridge(random_state=42)
    },
    "lasso": {
        "family": "linear",
        "estimator": Lasso(max_iter=20000, random_state=42)
    },
    "elastic_net": {
        "family": "linear",
        "estimator": ElasticNet(max_iter=20000, random_state=42)
    },
    "random_forest": {
        "family": "tree",
        "estimator": RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        )
    },
    "extra_trees": {
        "family": "tree",
        "estimator": ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1
        )
    },
    "gradient_boosting": {
        "family": "tree",
        "estimator": GradientBoostingRegressor(random_state=42)
    },
    "hist_gradient_boosting": {
        "family": "tree",
        "estimator": HistGradientBoostingRegressor(random_state=42)
    }
}

In [ ]:
def make_model_pipeline(model_name, model_specs, feature_views):
    spec = model_specs[model_name]
    family = spec["family"]

    pipe = Pipeline([
        ("preprocess", make_preprocessor(feature_views[family])),
        ("model", deepcopy(spec["estimator"]))
    ])
    return pipe


BASE_PIPELINES = {
    model_name: make_model_pipeline(model_name, MODEL_SPECS, FEATURE_VIEWS)
    for model_name in MODEL_SPECS
}

print("Base pipelines ready:")
print(list(BASE_PIPELINES.keys()))

In [ ]:
def run_pipeline_smoke_test(pipelines, X_train, y_train, X_valid, X_test):
    smoke_rows = []

    for model_name, pipe in pipelines.items():
        pipe.fit(X_train, y_train)

        Xt_train = pipe.named_steps["preprocess"].transform(X_train)
        Xt_valid = pipe.named_steps["preprocess"].transform(X_valid)
        Xt_test  = pipe.named_steps["preprocess"].transform(X_test)

        smoke_rows.append({
            "model": model_name,
            "train_shape": Xt_train.shape,
            "valid_shape": Xt_valid.shape,
            "test_shape": Xt_test.shape
        })

    smoke_df = pd.DataFrame(smoke_rows)
    return smoke_df


smoke_test_results = run_pipeline_smoke_test(
    pipelines=BASE_PIPELINES,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    X_test=X_test
)

display(smoke_test_results)

Step7: Validation

In [ ]:
def regression_metrics(y_true, y_pred):
    MSE = mean_squared_error(y_true, y_pred)
    RMSE = np.sqrt(MSE)
    
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": RMSE,
        "R2": r2_score(y_true, y_pred)
    }


def evaluate_pipeline(pipe, X_train, y_train, X_valid, y_valid, log_target=False):
    model = clone(pipe)

    if log_target:
        y_train_fit = np.log1p(y_train)
        model.fit(X_train, y_train_fit)

        pred_train = np.expm1(model.predict(X_train))
        pred_valid = np.expm1(model.predict(X_valid))
    else:
        model.fit(X_train, y_train)

        pred_train = model.predict(X_train)
        pred_valid = model.predict(X_valid)

    # optional safety for negative predictions
    pred_train = np.clip(pred_train, a_min=0, a_max=None)
    pred_valid = np.clip(pred_valid, a_min=0, a_max=None)

    train_metrics = regression_metrics(y_train, pred_train)
    valid_metrics = regression_metrics(y_valid, pred_valid)

    out = {}
    out.update({f"train_{k}": v for k, v in train_metrics.items()})
    out.update({f"valid_{k}": v for k, v in valid_metrics.items()})
    out["fitted_pipeline"] = model
    out["train_pred"] = pred_train
    out["valid_pred"] = pred_valid

    return out


def run_benchmark(pipelines, X_train, y_train, X_valid, y_valid, log_target=False):
    rows = []
    fitted_models = {}

    for model_name, pipe in pipelines.items():
        result = evaluate_pipeline(
            pipe=pipe,
            X_train=X_train,
            y_train=y_train,
            X_valid=X_valid,
            y_valid=y_valid,
            log_target=log_target
        )

        rows.append({
            "model": model_name,
            "target_scale": "log1p(Premium)" if log_target else "Premium",
            "train_MAE": result["train_MAE"],
            "valid_MAE": result["valid_MAE"],
            "train_RMSE": result["train_RMSE"],
            "valid_RMSE": result["valid_RMSE"],
            "train_R2": result["train_R2"],
            "valid_R2": result["valid_R2"]
        })

        fitted_models[model_name] = result["fitted_pipeline"]

    results_df = pd.DataFrame(rows)

    return results_df, fitted_models

In [ ]:
benchmark_raw_df, fitted_raw_models = run_benchmark(
    pipelines=BASE_PIPELINES,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    log_target=False
)

benchmark_raw_df = benchmark_raw_df.sort_values(
    by=["valid_RMSE", "valid_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("===== RAW TARGET BENCHMARK: Premium =====")
display(benchmark_raw_df)

In [ ]:
benchmark_log_df, fitted_log_models = run_benchmark(
    pipelines=BASE_PIPELINES,
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    log_target=True
)

benchmark_log_df = benchmark_log_df.sort_values(
    by=["valid_RMSE", "valid_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("===== LOG TARGET BENCHMARK: log1p(Premium) =====")
display(benchmark_log_df)

In [ ]:
benchmark_all_df = pd.concat(
    [benchmark_raw_df, benchmark_log_df],
    axis=0,
    ignore_index=True
)

benchmark_all_df = benchmark_all_df.sort_values(
    by=["valid_RMSE", "valid_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("===== COMBINED VALIDATION COMPARISON =====")
display(benchmark_all_df)

In [ ]:
gap_df = benchmark_all_df.copy()
gap_df["MAE_gap"] = gap_df["valid_MAE"] - gap_df["train_MAE"]
gap_df["RMSE_gap"] = gap_df["valid_RMSE"] - gap_df["train_RMSE"]
gap_df["R2_gap"] = gap_df["train_R2"] - gap_df["valid_R2"]

gap_df = gap_df.sort_values(by="valid_RMSE", ascending=True).reset_index(drop=True)

print("===== GENERALIZATION GAP DIAGNOSTICS =====")
display(
    gap_df[
        [
            "model", "target_scale",
            "train_MAE", "valid_MAE", "MAE_gap",
            "train_RMSE", "valid_RMSE", "RMSE_gap",
            "train_R2", "valid_R2", "R2_gap"
        ]
    ]
)

In [ ]:
raw_cmp = benchmark_raw_df[["model", "valid_MAE", "valid_RMSE", "valid_R2"]].copy()
raw_cmp = raw_cmp.rename(columns={
    "valid_MAE": "raw_valid_MAE",
    "valid_RMSE": "raw_valid_RMSE",
    "valid_R2": "raw_valid_R2"
})

log_cmp = benchmark_log_df[["model", "valid_MAE", "valid_RMSE", "valid_R2"]].copy()
log_cmp = log_cmp.rename(columns={
    "valid_MAE": "log_valid_MAE",
    "valid_RMSE": "log_valid_RMSE",
    "valid_R2": "log_valid_R2"
})

raw_vs_log_df = raw_cmp.merge(log_cmp, on="model", how="inner")

raw_vs_log_df["MAE_improvement_from_log"] = raw_vs_log_df["raw_valid_MAE"] - raw_vs_log_df["log_valid_MAE"]
raw_vs_log_df["RMSE_improvement_from_log"] = raw_vs_log_df["raw_valid_RMSE"] - raw_vs_log_df["log_valid_RMSE"]
raw_vs_log_df["R2_improvement_from_log"] = raw_vs_log_df["log_valid_R2"] - raw_vs_log_df["raw_valid_R2"]

raw_vs_log_df = raw_vs_log_df.sort_values(
    by="RMSE_improvement_from_log",
    ascending=False
).reset_index(drop=True)

print("===== RAW vs LOG TARGET COMPARISON =====")
display(raw_vs_log_df)

Step8: Chronological CV:

In [ ]:
def make_expanding_date_splits(date_series, n_splits=4, min_train_frac=0.50):
    """
    date_series: pd.Series of outer-train dates, already aligned to X_train_tune rows
    Returns list of (train_idx, valid_idx) positional splits for X_train_tune.iloc[]
    """

    ds = pd.to_datetime(date_series).reset_index(drop=True)
    unique_dates = np.array(sorted(ds.dropna().unique()))

    if len(unique_dates) < (n_splits + 1):
        raise ValueError("Not enough unique dates in training block for the requested number of splits.")

    min_train_dates = max(1, int(len(unique_dates) * min_train_frac))

    candidate_positions = np.linspace(
        min_train_dates,
        len(unique_dates) - 1,
        n_splits + 1,
        dtype=int
    )[1:]

    candidate_positions = np.unique(candidate_positions)

    splits = []
    for cut_pos in candidate_positions:
        train_end_date = unique_dates[cut_pos - 1]
        valid_date = unique_dates[cut_pos]

        train_idx = np.where(ds <= train_end_date)[0]
        valid_idx = np.where(ds == valid_date)[0]

        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue

        splits.append((train_idx, valid_idx))

    if len(splits) == 0:
        raise ValueError("Failed to create any valid inner chronological splits.")

    return splits

In [ ]:
def evaluate_param_set_cv(
    base_pipe,
    param_dict,
    X_train,
    y_train,
    inner_splits,
    log_target=False
):
    fold_rows = []

    for fold_id, (tr_idx, va_idx) in enumerate(inner_splits, start=1):
        X_tr = X_train.iloc[tr_idx]
        X_va = X_train.iloc[va_idx]
        y_tr = y_train.iloc[tr_idx]
        y_va = y_train.iloc[va_idx]

        pipe = clone(base_pipe)
        pipe.set_params(**param_dict)

        if log_target:
            y_tr_fit = np.log1p(y_tr)
            pipe.fit(X_tr, y_tr_fit)
            pred_va = np.expm1(pipe.predict(X_va))
        else:
            pipe.fit(X_tr, y_tr)
            pred_va = pipe.predict(X_va)

        pred_va = np.clip(pred_va, a_min=0, a_max=None)
        metrics = regression_metrics(y_va, pred_va)

        fold_rows.append({
            "fold": fold_id,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"]
        })

    fold_df = pd.DataFrame(fold_rows)

    summary = {
        "cv_MAE_mean": fold_df["MAE"].mean(),
        "cv_RMSE_mean": fold_df["RMSE"].mean(),
        "cv_R2_mean": fold_df["R2"].mean(),
        "cv_MAE_std": fold_df["MAE"].std(),
        "cv_RMSE_std": fold_df["RMSE"].std(),
        "cv_R2_std": fold_df["R2"].std()
    }

    return summary, fold_df

In [ ]:
def random_search_time_cv(
    base_pipe,
    param_distributions,
    X_train,
    y_train,
    inner_splits,
    n_iter=20,
    random_state=42,
    log_target=False,
    verbose=True
):
    sampled_params = list(
        ParameterSampler(
            param_distributions=param_distributions,
            n_iter=n_iter,
            random_state=random_state
        )
    )

    rows = []

    for i, param_dict in enumerate(sampled_params, start=1):
        summary, _ = evaluate_param_set_cv(
            base_pipe=base_pipe,
            param_dict=param_dict,
            X_train=X_train,
            y_train=y_train,
            inner_splits=inner_splits,
            log_target=log_target
        )

        row = {
            "trial": i,
            "params": param_dict,
            **summary
        }
        rows.append(row)

        if verbose:
            print(
                f"[{i:02d}/{n_iter}] "
                f"RMSE={summary['cv_RMSE_mean']:.4f} | "
                f"MAE={summary['cv_MAE_mean']:.4f} | "
                f"R2={summary['cv_R2_mean']:.4f}"
            )

    results_df = pd.DataFrame(rows).sort_values(
        by=["cv_RMSE_mean", "cv_MAE_mean"],
        ascending=[True, True]
    ).reset_index(drop=True)

    best_params = results_df.loc[0, "params"]
    return results_df, best_params

In [ ]:
def fit_and_score_outer_validation(
    base_pipe,
    best_params,
    X_train,
    y_train,
    X_valid,
    y_valid,
    log_target=False
):
    pipe = clone(base_pipe)
    pipe.set_params(**best_params)

    if log_target:
        y_train_fit = np.log1p(y_train)
        pipe.fit(X_train, y_train_fit)

        pred_train = np.expm1(pipe.predict(X_train))
        pred_valid = np.expm1(pipe.predict(X_valid))
    else:
        pipe.fit(X_train, y_train)

        pred_train = pipe.predict(X_train)
        pred_valid = pipe.predict(X_valid)

    pred_train = np.clip(pred_train, a_min=0, a_max=None)
    pred_valid = np.clip(pred_valid, a_min=0, a_max=None)

    train_metrics = regression_metrics(y_train, pred_train)
    valid_metrics = regression_metrics(y_valid, pred_valid)

    return {
        "fitted_pipeline": pipe,
        "best_params": best_params,
        "train_MAE": train_metrics["MAE"],
        "train_RMSE": train_metrics["RMSE"],
        "train_R2": train_metrics["R2"],
        "valid_MAE": valid_metrics["MAE"],
        "valid_RMSE": valid_metrics["RMSE"],
        "valid_R2": valid_metrics["R2"]
    }

In [ ]:
# Inner chronological splits from TRAIN dates
X_train_tune = X_train.reset_index(drop=True).copy()
y_train_tune = y_train.reset_index(drop=True).copy()
train_dates = pd.to_datetime(train_df["Date_last_renewal"]).reset_index(drop=True).copy()

# safety checks
assert len(X_train_tune) == len(y_train_tune) == len(train_dates), \
    "X_train, y_train, and train_df[Date_last_renewal] must align in length."

inner_splits = make_expanding_date_splits(
    date_series=train_dates,
    n_splits=4,
    min_train_frac=0.50
)

print(f"Number of inner splits: {len(inner_splits)}")
for i, (tr_idx, va_idx) in enumerate(inner_splits, start=1):
    tr_dates = train_dates.iloc[tr_idx]
    va_dates = train_dates.iloc[va_idx]
    print(
        f"Fold {i}: "
        f"train={len(tr_idx)} rows ({tr_dates.min().date()} to {tr_dates.max().date()}), "
        f"valid={len(va_idx)} rows ({va_dates.min().date()} to {va_dates.max().date()})"
    )

In [ ]:
TUNING_SPECS = {
    "extra_trees": {
        "pipeline": BASE_PIPELINES["extra_trees"],
        "param_distributions": {
            "model__n_estimators": randint(300, 1001),
            "model__max_depth": [None, 6, 10, 14, 18, 24],
            "model__min_samples_split": randint(2, 21),
            "model__min_samples_leaf": randint(1, 11),
            "model__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8, 1.0],
            "model__bootstrap": [False, True]
        }
    },
    "random_forest": {
        "pipeline": BASE_PIPELINES["random_forest"],
        "param_distributions": {
            "model__n_estimators": randint(300, 1001),
            "model__max_depth": [None, 6, 10, 14, 18, 24],
            "model__min_samples_split": randint(2, 21),
            "model__min_samples_leaf": randint(1, 11),
            "model__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8, 1.0],
            "model__bootstrap": [True, False]
        }
    },
    "hist_gradient_boosting": {
        "pipeline": BASE_PIPELINES["hist_gradient_boosting"],
        "param_distributions": {
            "model__learning_rate": uniform(0.02, 0.18),
            "model__max_iter": randint(200, 801),
            "model__max_depth": [None, 3, 5, 7, 9],
            "model__min_samples_leaf": randint(10, 81),
            "model__l2_regularization": uniform(0.0, 1.0),
            "model__max_bins": randint(64, 256)
        }
    }
}

In [ ]:
tuning_results_raw = {}
outer_valid_results_raw = []

for model_name, spec in TUNING_SPECS.items():
    print(f"\n===== TUNING {model_name.upper()} | RAW TARGET =====")

    search_df, best_params = random_search_time_cv(
        base_pipe=spec["pipeline"],
        param_distributions=spec["param_distributions"],
        X_train=X_train_tune,
        y_train=y_train_tune,
        inner_splits=inner_splits,
        n_iter=20,
        random_state=42,
        log_target=False,
        verbose=True
    )

    tuning_results_raw[model_name] = {
        "search_df": search_df,
        "best_params": best_params
    }

    outer_result = fit_and_score_outer_validation(
        base_pipe=spec["pipeline"],
        best_params=best_params,
        X_train=X_train_tune,
        y_train=y_train_tune,
        X_valid=X_valid,
        y_valid=y_valid,
        log_target=False
    )

    outer_valid_results_raw.append({
        "model": model_name,
        "target_scale": "Premium",
        "best_params": best_params,
        "train_MAE": outer_result["train_MAE"],
        "valid_MAE": outer_result["valid_MAE"],
        "train_RMSE": outer_result["train_RMSE"],
        "valid_RMSE": outer_result["valid_RMSE"],
        "train_R2": outer_result["train_R2"],
        "valid_R2": outer_result["valid_R2"]
    })

tuned_raw_df = pd.DataFrame(outer_valid_results_raw).sort_values(
    by=["valid_RMSE", "valid_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("\n===== TUNED MODELS | OUTER VALIDATION | RAW TARGET =====")
display(tuned_raw_df)

In [ ]:
tuning_results_log = {}
outer_valid_results_log = []

for model_name, spec in TUNING_SPECS.items():
    print(f"\n===== TUNING {model_name.upper()} | LOG TARGET =====")

    search_df, best_params = random_search_time_cv(
        base_pipe=spec["pipeline"],
        param_distributions=spec["param_distributions"],
        X_train=X_train_tune,
        y_train=y_train_tune,
        inner_splits=inner_splits,
        n_iter=20,
        random_state=42,
        log_target=True,
        verbose=True
    )

    tuning_results_log[model_name] = {
        "search_df": search_df,
        "best_params": best_params
    }

    outer_result = fit_and_score_outer_validation(
        base_pipe=spec["pipeline"],
        best_params=best_params,
        X_train=X_train_tune,
        y_train=y_train_tune,
        X_valid=X_valid,
        y_valid=y_valid,
        log_target=True
    )

    outer_valid_results_log.append({
        "model": model_name,
        "target_scale": "log1p(Premium)",
        "best_params": best_params,
        "train_MAE": outer_result["train_MAE"],
        "valid_MAE": outer_result["valid_MAE"],
        "train_RMSE": outer_result["train_RMSE"],
        "valid_RMSE": outer_result["valid_RMSE"],
        "train_R2": outer_result["train_R2"],
        "valid_R2": outer_result["valid_R2"]
    })

tuned_log_df = pd.DataFrame(outer_valid_results_log).sort_values(
    by=["valid_RMSE", "valid_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("\n===== TUNED MODELS | OUTER VALIDATION | LOG TARGET =====")
display(tuned_log_df)

In [ ]:
tuned_all_df = pd.concat([tuned_raw_df, tuned_log_df], axis=0, ignore_index=True)

tuned_all_df["MAE_gap"] = tuned_all_df["valid_MAE"] - tuned_all_df["train_MAE"]
tuned_all_df["RMSE_gap"] = tuned_all_df["valid_RMSE"] - tuned_all_df["train_RMSE"]
tuned_all_df["R2_gap"] = tuned_all_df["train_R2"] - tuned_all_df["valid_R2"]

tuned_all_df = tuned_all_df.sort_values(
    by=["valid_RMSE", "valid_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("===== FINAL TUNED COMPARISON ON OUTER VALIDATION =====")
display(
    tuned_all_df[
        [
            "model", "target_scale",
            "train_MAE", "valid_MAE", "MAE_gap",
            "train_RMSE", "valid_RMSE", "RMSE_gap",
            "train_R2", "valid_R2", "R2_gap",
            "best_params"
        ]
    ]
)

Step9: Test Evaluation

In [ ]:
def fit_final_and_evaluate(
    base_pipe,
    best_params,
    X_train_full,
    y_train_full,
    X_test,
    y_test,
    log_target=False
):
    pipe = clone(base_pipe)
    pipe.set_params(**best_params)

    if log_target:
        y_train_fit = np.log1p(y_train_full)
        pipe.fit(X_train_full, y_train_fit)

        pred_train_full = np.expm1(pipe.predict(X_train_full))
        pred_test = np.expm1(pipe.predict(X_test))
    else:
        pipe.fit(X_train_full, y_train_full)

        pred_train_full = pipe.predict(X_train_full)
        pred_test = pipe.predict(X_test)

    pred_train_full = np.clip(pred_train_full, a_min=0, a_max=None)
    pred_test = np.clip(pred_test, a_min=0, a_max=None)

    train_full_metrics = regression_metrics(y_train_full, pred_train_full)
    test_metrics = regression_metrics(y_test, pred_test)

    return {
        "fitted_pipeline": pipe,
        "pred_train_full": pred_train_full,
        "pred_test": pred_test,
        "train_full_MAE": train_full_metrics["MAE"],
        "train_full_RMSE": train_full_metrics["RMSE"],
        "train_full_R2": train_full_metrics["R2"],
         "test_MAE": test_metrics["MAE"],
        "test_RMSE": test_metrics["RMSE"],
        "test_R2": test_metrics["R2"]
    }

In [ ]:
X_train_full = pd.concat([X_train, X_valid], axis=0).reset_index(drop=True)
y_train_full = pd.concat([y_train, y_valid], axis=0).reset_index(drop=True)

print("Final training set shape:", X_train_full.shape)
print("Test set shape:", X_test.shape)

In [ ]:
FINAL_CANDIDATES = [
    {
        "model": "random_forest",
        "target_scale": "log1p(Premium)",
        "base_pipe": BASE_PIPELINES["random_forest"],
        "best_params": tuning_results_log["random_forest"]["best_params"],
        "log_target": True
    },
    {
        "model": "extra_trees",
        "target_scale": "Premium",
        "base_pipe": BASE_PIPELINES["extra_trees"],
        "best_params": tuning_results_raw["extra_trees"]["best_params"],
        "log_target": False
    }
]

In [ ]:
final_test_rows = []
final_fitted_models = {}

for spec in FINAL_CANDIDATES:
    print(f"\n===== FINAL TEST EVALUATION: {spec['model']} | {spec['target_scale']} =====")

    result = fit_final_and_evaluate(
        base_pipe=spec["base_pipe"],
        best_params=spec["best_params"],
        X_train_full=X_train_full,
        y_train_full=y_train_full,
        X_test=X_test,
        y_test=y_test,
        log_target=spec["log_target"]
    )

    final_fitted_models[f"{spec['model']}__{spec['target_scale']}"] = result["fitted_pipeline"]

    final_test_rows.append({
        "model": spec["model"],
        "target_scale": spec["target_scale"],
        "train_full_MAE": result["train_full_MAE"],
        "test_MAE": result["test_MAE"],
        "MAE_gap": result["test_MAE"] - result["train_full_MAE"],
        "train_full_RMSE": result["train_full_RMSE"],
        "test_RMSE": result["test_RMSE"],
        "RMSE_gap": result["test_RMSE"] - result["train_full_RMSE"],
        "train_full_R2": result["train_full_R2"],
        "test_R2": result["test_R2"],
        "R2_gap": result["train_full_R2"] - result["test_R2"],
        "best_params": spec["best_params"]
    })

final_test_df = pd.DataFrame(final_test_rows).sort_values(
    by=["test_RMSE", "test_MAE"],
    ascending=[True, True]
).reset_index(drop=True)

print("\n===== FINAL OUT-OF-TIME TEST RESULTS =====")
display(final_test_df)

In [ ]:
final_winner = final_test_df.iloc[0].copy()

print("===== FINAL SELECTED MODEL =====")
display(final_winner.to_frame().T)

In [ ]:
winner_model_name = final_winner["model"]
winner_target_scale = final_winner["target_scale"]

winner_spec = None
for spec in FINAL_CANDIDATES:
    if spec["model"] == winner_model_name and spec["target_scale"] == winner_target_scale:
        winner_spec = spec
        break

winner_result = fit_final_and_evaluate(
    base_pipe=winner_spec["base_pipe"],
    best_params=winner_spec["best_params"],
    X_train_full=X_train_full,
    y_train_full=y_train_full,
    X_test=X_test,
    y_test=y_test,
    log_target=winner_spec["log_target"]
)

test_pred = winner_result["pred_test"]
test_actual = y_test.reset_index(drop=True).copy()

test_diag_df = pd.DataFrame({
    "actual": test_actual,
    "pred": test_pred,
    "residual": test_actual - test_pred,
    "abs_error": np.abs(test_actual - test_pred)
})

print("===== TEST ERROR SUMMARY =====")
display(test_diag_df[["actual", "pred", "residual", "abs_error"]].describe())

In [ ]:
test_diag_df["actual_bucket"] = pd.qcut(
    test_diag_df["actual"],
    q=5,
    duplicates="drop"
)

bucket_error_df = (
    test_diag_df
    .groupby("actual_bucket", observed=False)
    .agg(
        n=("actual", "size"),
        actual_mean=("actual", "mean"),
        pred_mean=("pred", "mean"),
        mae=("abs_error", "mean"),
        rmse=("residual", lambda x: np.sqrt(np.mean(np.square(x))))
    )
    .reset_index()
)

print("===== TEST ERROR BY ACTUAL PREMIUM BUCKET =====")
display(bucket_error_df)

Step10: Feature Importance 

In [ ]:
best_pipe = winner_result["fitted_pipeline"]

preprocess = best_pipe.named_steps["preprocess"]
final_model = best_pipe.named_steps["model"]

print("Final pipeline:")
display(best_pipe)

print("\nUnderlying estimator:")
print(final_model)

In [ ]:
def get_feature_names_from_fitted_preprocessor(preprocessor):
    feature_names = []

    for name, transformer, cols in preprocessor.transformers_:
        if name == "remainder" and transformer == "drop":
            continue

        # Pipeline branch
        if hasattr(transformer, "named_steps"):
            last_step = list(transformer.named_steps.values())[-1]
        else:
            last_step = transformer

        if hasattr(last_step, "get_feature_names_out"):
            try:
                names = last_step.get_feature_names_out(cols)
            except:
                names = last_step.get_feature_names_out()
        else:
            names = cols

        feature_names.extend(list(names))

    return feature_names


transformed_feature_names = get_feature_names_from_fitted_preprocessor(preprocess)

print("Number of transformed features:", len(transformed_feature_names))
print(transformed_feature_names[:20])

In [ ]:
X_test_trans = preprocess.transform(X_test)

if hasattr(X_test_trans, "toarray"):
    X_test_trans_dense = X_test_trans.toarray()
else:
    X_test_trans_dense = X_test_trans

X_test_trans_df = pd.DataFrame(
    X_test_trans_dense,
    columns=transformed_feature_names,
    index=X_test.index
)

print("Transformed test shape:", X_test_trans_df.shape)
display(X_test_trans_df.head())

In [ ]:
def raw_scale_mae_from_pipeline(estimator, X, y_true):
    """
    Uses winner_target_scale from your notebook.
    If model was trained on log1p(Premium), convert predictions back.
    """
    pred = estimator.predict(X)

    if winner_target_scale == "log1p(Premium)":
        pred = np.expm1(pred)

    pred = np.clip(pred, a_min=0, a_max=None)

    return -mean_absolute_error(y_true, pred)


perm_result = permutation_importance(
    estimator=best_pipe,
    X=X_test,
    y=y_test,
    scoring=make_scorer(
        lambda yt, yp: -mean_absolute_error(
            yt,
            np.clip(np.expm1(yp), a_min=0, a_max=None)
        ) if winner_target_scale == "log1p(Premium)"
        else -mean_absolute_error(yt, np.clip(yp, a_min=0, a_max=None)),
        greater_is_better=True
    ),
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values("importance_mean", ascending=False)

print("===== PERMUTATION IMPORTANCE (ORIGINAL FEATURE LEVEL) =====")
display(perm_importance_df.head(25))

In [ ]:
top_k = 20
plot_df = perm_importance_df.head(top_k).sort_values("importance_mean")

plt.figure(figsize=(10, 7))
plt.barh(plot_df["feature"], plot_df["importance_mean"], xerr=plot_df["importance_std"])
plt.xlabel("Importance (drop in score after permutation)")
plt.ylabel("Original feature")
plt.title(f"Top {top_k} Permutation Importances")
plt.tight_layout()
plt.show()

In [ ]:
# sample for SHAP
shap_sample_n = 300
X_shap = X_test_trans_df.sample(shap_sample_n, random_state=42)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_shap)

print("SHAP array shape:", np.array(shap_values).shape)

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    plot_type="bar",
    max_display=20
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    max_display=20
)

In [ ]:
original_features = list(X_test.columns)

def map_transformed_feature_to_original(transformed_name, original_features):
    """
    Example:
    Type_fuel_Diesel -> Type_fuel
    Payment_Monthly  -> Payment
    vehicle_value_log -> vehicle_value_log
    """
    matches = [
        col for col in original_features
        if transformed_name == col or transformed_name.startswith(col + "_")
    ]
    if matches:
        return max(matches, key=len)
    return transformed_name


transformed_to_original = {
    col: map_transformed_feature_to_original(col, original_features)
    for col in X_shap.columns
}

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_detail_df = pd.DataFrame({
    "transformed_feature": X_shap.columns,
    "original_feature": [transformed_to_original[col] for col in X_shap.columns],
    "mean_abs_shap": mean_abs_shap
})

grouped_shap_df = (
    shap_detail_df
    .groupby("original_feature", as_index=False)["mean_abs_shap"]
    .sum()
    .sort_values("mean_abs_shap", ascending=False)
)

print("===== GROUPED SHAP IMPORTANCE (ORIGINAL FEATURE LEVEL) =====")
display(grouped_shap_df.head(25))

In [ ]:
top_k = 20
plot_df = grouped_shap_df.head(top_k).sort_values("mean_abs_shap")

plt.figure(figsize=(10, 7))
plt.barh(plot_df["original_feature"], plot_df["mean_abs_shap"])
plt.xlabel("Mean |SHAP value|")
plt.ylabel("Original feature")
plt.title(f"Top {top_k} Grouped SHAP Importances")
plt.tight_layout()
plt.show()

In [ ]:
features_to_check = [
    "vehicle_value_log",
    "Power",
    "vehicle_age",
    "value_per_vehicle_year",
    "policy_headroom",
    "age_at_renewal",
    "licence_tenure",
    "flag_young_driver",
    "flag_newly_licensed",
    "flag_n_doors_zero",
    "flag_power_missing",
    "flag_type_fuel_missing",
    "flag_length_missing"
]

for feat in features_to_check:
    if feat in X_shap.columns:
        shap.dependence_plot(
            feat,
            shap_values,
            X_shap,
            interaction_index="auto"
        )

In [ ]:
row_idx = X_shap.index[0]
row_pos = X_shap.index.get_loc(row_idx)

print("Selected row index:", row_idx)
print("Actual Premium:", y_test.loc[row_idx])

raw_pred = best_pipe.predict(X_test.loc[[row_idx]])[0]
if winner_target_scale == "log1p(Premium)":
    raw_pred = np.expm1(raw_pred)

raw_pred = np.clip(raw_pred, a_min=0, a_max=None)
print("Predicted Premium:", raw_pred)

In [ ]:
base_value = explainer.expected_value

shap_explanation = shap.Explanation(
    values=shap_values[row_pos],
    base_values=base_value,
    data=X_shap.iloc[row_pos].values,
    feature_names=X_shap.columns.tolist()
)

shap.plots.waterfall(shap_explanation)

In [ ]:
importance_compare_df = (
    perm_importance_df[["feature", "importance_mean"]]
    .rename(columns={
        "feature": "original_feature",
        "importance_mean": "permutation_importance"
    })
    .merge(
        grouped_shap_df.rename(columns={"mean_abs_shap": "grouped_shap_importance"}),
        on="original_feature",
        how="outer"
    )
    .sort_values(
        ["grouped_shap_importance", "permutation_importance"],
        ascending=False
    )
)

print("===== IMPORTANCE COMPARISON =====")
display(importance_compare_df.head(25))

In [ ]:
# Special Cases
test_pred = best_pipe.predict(X_test)

if winner_target_scale == "log1p(Premium)":
    test_pred = np.expm1(test_pred)

test_pred = np.clip(test_pred, a_min=0, a_max=None)

case_df = X_test.copy()
case_df["actual_premium"] = y_test
case_df["predicted_premium"] = test_pred
case_df["residual"] = case_df["predicted_premium"] - case_df["actual_premium"]
case_df["abs_error"] = np.abs(case_df["residual"])

print(case_df[["actual_premium", "predicted_premium", "residual", "abs_error"]].describe())

In [ ]:
high_cutoff = case_df["actual_premium"].quantile(0.90)

high_under_df = case_df[case_df["actual_premium"] >= high_cutoff].copy()
high_under_df = high_under_df.sort_values("residual", ascending=True)   # most negative residual first

high_under_idx = high_under_df.index[0]

# low-premium overprediction:
# among the bottom 25% actual premiums, choose the most overpredicted row
low_cutoff = case_df["actual_premium"].quantile(0.25)

low_over_df = case_df[case_df["actual_premium"] <= low_cutoff].copy()
low_over_df = low_over_df.sort_values("residual", ascending=False)      # most positive residual first

low_over_idx = low_over_df.index[0]

print("High-premium underprediction case index:", high_under_idx)
display(case_df.loc[[high_under_idx], ["actual_premium", "predicted_premium", "residual", "abs_error"]])

print("Low-premium overprediction case index:", low_over_idx)
display(case_df.loc[[low_over_idx], ["actual_premium", "predicted_premium", "residual", "abs_error"]])

In [ ]:
selected_case_idx = [high_under_idx, low_over_idx]

X_cases = X_test.loc[selected_case_idx].copy()
X_cases_trans = preprocess.transform(X_cases)

if hasattr(X_cases_trans, "toarray"):
    X_cases_trans = X_cases_trans.toarray()

X_cases_trans_df = pd.DataFrame(
    X_cases_trans,
    columns=transformed_feature_names,
    index=X_cases.index
)

display(X_cases_trans_df.head())

In [ ]:
local_explainer = shap.TreeExplainer(final_model)
local_shap_values = local_explainer.shap_values(X_cases_trans_df, check_additivity=False)

print("Local SHAP shape:", np.array(local_shap_values).shape)

In [ ]:
# Waterfall: high-premium underprediction case
row_pos = X_cases_trans_df.index.get_loc(high_under_idx)

print("===== HIGH-PREMIUM UNDERPREDICTION CASE =====")
print("Index:", high_under_idx)
print("Actual Premium:", case_df.loc[high_under_idx, "actual_premium"])
print("Predicted Premium:", case_df.loc[high_under_idx, "predicted_premium"])
print("Residual (pred - actual):", case_df.loc[high_under_idx, "residual"])

high_under_explanation = shap.Explanation(
    values=local_shap_values[row_pos],
    base_values=local_explainer.expected_value,
    data=X_cases_trans_df.loc[high_under_idx].values,
    feature_names=X_cases_trans_df.columns.tolist()
)

shap.plots.waterfall(high_under_explanation, max_display=15)

In [ ]:
# Waterfall: low-premium underprediction case
row_pos = X_cases_trans_df.index.get_loc(low_over_idx)

print("===== LOW-PREMIUM OVERPREDICTION CASE =====")
print("Index:", low_over_idx)
print("Actual Premium:", case_df.loc[low_over_idx, "actual_premium"])
print("Predicted Premium:", case_df.loc[low_over_idx, "predicted_premium"])
print("Residual (pred - actual):", case_df.loc[low_over_idx, "residual"])

low_over_explanation = shap.Explanation(
    values=local_shap_values[row_pos],
    base_values=local_explainer.expected_value,
    data=X_cases_trans_df.loc[low_over_idx].values,
    feature_names=X_cases_trans_df.columns.tolist()
)

shap.plots.waterfall(low_over_explanation, max_display=15)

In [ ]:
context_cols = [
    "Payment",
    "Type_risk",
    "Second_driver",
    "Value_vehicle",
    "vehicle_value_log",
    "value_per_vehicle_year",
    "vehicle_age",
    "policy_tenure",
    "licence_tenure",
    "flag_n_doors_zero",
    "flag_young_driver",
    "flag_newly_licensed"
]

valid_context_cols = [c for c in context_cols if c in X_test.columns]

print("===== ORIGINAL FEATURES: HIGH-PREMIUM UNDERPREDICTION =====")
display(X_test.loc[[high_under_idx], valid_context_cols])

print("===== ORIGINAL FEATURES: LOW-PREMIUM OVERPREDICTION =====")
display(X_test.loc[[low_over_idx], valid_context_cols])